# 101 - How prysm Works

This notebook describes the basics of how the library is structured.  It
will help the reader understand the key concepts of the library, and how
it differs from similar modules.  We'll show that:

- The code closely follows the physics and there is no magic or unexpected
  computation
- There is generally one or two extra function calls to explicitly build
  grids and similar, without abstract notions of sampling passed around
- See why prysm is mostly _functions_ with relatively few classes, and what
  that buys you.
- Learn when full grids are used vs just a `dx` parameter carried, and what
  the units of things are

The classes following this one are:

- [102 notebook]('./102 - coordinates, grids, and units') dives deeper
into coordinates, grids and units.
The rest of the Foundations college drills into the pieces this notebook
gestures at: grids and geometry (102), the `RichData` container and IO (103),
and backends and precision (104).

## Imports

prysm is split into many submodules.  The common practice is to import the
pieces you need and _not_ use star imports.  There is not a top level namespace
with the hundreds of functions and classes exported for you to `from prysm import *` from.

In [ ]:
# import just the pieces to "lift" them into your main namespace
from prysm.coordinates import make_xy_grid, cart_to_polar
from prysm.geometry import circle
from prysm.polynomials import zernike_nm, zernike_nm_seq, noll_to_nm, sum_of_2d_modes

import numpy as np
from matplotlib import pyplot as plt

# or use the modules and dot syntax
# from prysm import (
#     coordinates,
#     geometry,
#     polynomials,
#     propagation
# )
# coordinates.make_xy_grid; coordinates.cart_to_polar, etc.

## Everything operates on grids you own

Every function in prysm takes the relevant coordinates as arguments - `x` and
`y`, or `rho` and `theta` - and returns ordinary arrays.  No function takes a
`sample_count` or `npix` argument.  The con to this is you creating the grid
in one or two lines of additional code, every time, while the pros are:

- the grid can never get desynced between functions (size mismatch, FFT-centered vs symmetric, etc)
- all the pieces of the library can just operate on simple arrays of points
- there is no performance impact to recreating the grid everywhere and repeatedly
- there is no hidden cache mechanism to deal with the above point while having an abstract grid API

`make_xy_grid` and `cart_to_polar` build a grid, returning ordinary 2d numpy arrays (unless you are using a non-numpy backend, see class 104 in this college) TODO: update 104 link:

In [ ]:
x, y = make_xy_grid(256, diameter=2)   # 256x256, spanning [-1, 1]
r, t = cart_to_polar(x, y)

focus = zernike_nm(2, 0, r, t)         # defocus over the unit disk

fig, ax = plt.subplots()
im = ax.imshow(focus, extent=[x.min(), x.max(), y.min(), y.max()], cmap='inferno')
fig.colorbar(im, ax=ax)
ax.set(title='Z(2,0) defocus')

Here you can see the next principle of nothing more than you asked for: the
zernike function did not set the array to some sentinal value (zero or NaN, say)
outside the orthogonal domain.  It just computed the actual Zernike polynomial
over the coordinates you gave.  In real models, the amplitude part of the problem
will zero what is going on out there anyway and it is completely fine to have that
"garbage" beyond r=1.  In fitting problems over imperfect domains, using a sentinal
value will produce _the wrong answer_ while leaving the non-orthogonal part of the
polynomial out there will only cause a small decrease in how well conditioned the
problem is, which the least squares fitting process will take care of without issue.

If your problem/model has several different planes (pupil and PSF, for example) you
simply have several different grids that you keep track of.

## Functions, not black boxes

prysm is predominantly composed of functions you combine, with relatively
few classes.  This is because simple functions with little to no coupling
between each other have an easy learning curve, and there is no _state_
to couple with this _computation_.  When state and computation naturally
are co-located such as with interferometric data, a wavefront in a propagation
model, or the optical system in a raytrace model, classes are used.

If you have used other physical-optics packages, you may have met a `Wavefront`
(PROPER) that is mutated as it moves through a system, or an `OpticalSystem`
(POPPY) that owns the whole problem.  prysm's `propagation.Wavefront` is closer
to PROPER, but has a less complicated set of options and switches.  You will
use a few more free functions compared to something like POPPY or HCIpy in
setting up the problem, but the actual propagation model is usually just a few
method calls on a `Wavefront`.

The decoupled nature of the library also leads to a smaller surface area.
POPPY has many (perhaps dozens) of wavefront subclasses for polynomial bases,
grid phases, etc.  Analogs of these things exist as free functions in their
own modules of prysm, but they come together through simple composition
of a phase array and amplitude array (or indeed, raw complex field if that
is what you have).  There is no coupling and there is no maze of subclasses.

In [ ]:
amplitude = circle(1, r)                       # 1 inside the unit disk, 0 outside
wfe = 0.25 * zernike_nm(2, 2, r, t)            # astigmatism, in waves

pupil = amplitude * np.exp(1j * 2 * np.pi * wfe)   # complex field

fig, axs = plt.subplots(ncols=2, figsize=(9, 4))
axs[0].imshow(np.abs(pupil), extent=[-1, 1, -1, 1], cmap='gray')
axs[0].set(title='amplitude')
axs[1].imshow(np.angle(pupil) * amplitude, extent=[-1, 1, -1, 1], cmap='RdBu')
axs[1].set(title='phase [rad, wrapped]')

The separation of creation of amplitude and phase, or the grid from amplitude
or phase, allows you to focus on just one step of the problem at a time.

Because there is no hidden state, anything slow that does not belong in a loop
is trivially kept out of the loop, and a natural "cache" forms in the variables
of your program with _no_ mechanical complexity.  For example, this 6 lines of
code builds two phase maps as weighed sums of zernikes with no hidden cache,
and the fast and repeated part (the weighted sum) separated from the slow part
that need not be repeated (building the basis functions):

In [ ]:
nms = [noll_to_nm(j) for j in range(4,11)]
basis = zernike_nm_seq(nms, r, t, norm=True)

coef1 = np.random.rand(len(nms))
coef2 = np.random.rand(nms)

wfe1 = sum_of_2d_modes(basis, coef1)
wfe2 = sum_of_2d_modes(basis, coef2)

## `dx`, or `x`?

Some constructors take `x, y`; others take `dx`.  prysm assumes rectilinear
sampling with `dy == dx`, although it does not require arrays be square like
many other libraries do.  It basically comes down to whether a computation
needs to know the actual vector of coordinates, or it just needs to know
what the spacing is.  Extra data is not passed around when it is not needed.

Likewise, if you want to know the physical size of something (a diameter, say)
you can produce the grid that way.  If you want to exactly match the grid from
other data (an interferometer measurement, a saved wavefront map from another
program, etc) you can build it from dx itself, which will be "FFT" aligned:

In [ ]:
# the same 256-sample, 2-unit-wide grid, two equivalent ways
x1, y1 = make_xy_grid(256, diameter=2)     # by total diameter
x2, y2 = make_xy_grid(256, dx=2/256)       # by sample spacing
print('same grid:', np.allclose(x1, x2))
print('dx =', float(x1[0, 1] - x1[0, 0]))

### Units

in general prysm does not require units to be fixed to some particular value,
but it does use a system of "preferred units" - which keep quantities relatively
close to 1 where possible.  As a general guide:

- Spatial coordinates in the pupil - mm
- Spatial coordinates in the psf - um
- Frequency coordinates for MTF/OTF - (cy/mm)
- wavelength - um

All that is actually "baked in" to the library is for example a factor of 1e3 between
pupil spatial and wavelength, or pupil spatial and psf spatial.  You could use inches
and milli-inches and it would make no difference.  Or meters and mm.

### Where to go next

- **102 - coordinates, grids, and units** builds out grids and the geometry
  (aperture) functions.
- **103 - RichData, plotting, and IO** covers the container class and reading and
  writing measured data.
- **104 - backends and precision** covers float32/float64 and running on a GPU.